# AI Pharmacist — RAG Notebook (Qwen 2.5 3B-Instruct)

Load a completely open, non-gated model, ask it a question, and use a RAG system so it answers accurately from our custom document instead of guessing.

Model used: `Qwen/Qwen2.5-3B-Instruct` (No Hugging Face token or license gate required!)
Knowledge base: `symptoms_and_medicines.txt` (50 symptoms x 10 medicines).

# How to use model and what youll need

1- youll need https://ollama.com/download   

2- you will run:   ollama run qwen2.5:3b (https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/tree/main)   ( to download the 2 gb model that we will use (uses 3gb vram) )

3- youll also need the MED TXT file that contains all the medicines and the symptoms for the RAG system




In [1]:
!pip install -q -U transformers accelerate sentencepiece
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 90.6 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-3B-Instruct"    # https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/tree/main

print("Downloading and loading Qwen 2.5 3B-Instruct...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Model loaded successfully!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [3]:
def generate_text(prompt, max_new_tokens=300):
    # Setting up system instructions for the Pharmacist identity
    messages = [
        {"role": "system", "content": "You are a helpful pharmacist assistant."},
        {"role": "user", "content": prompt},
    ]

    # Apply Qwen's chat template structure
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    # Isolate only the generated response text
    output_tokens = outputs[0][input_len:]
    return tokenizer.decode(output_tokens, skip_special_tokens=True)

### without RAG

In [4]:
question = "What medicine should I take for a headache?"
answer = generate_text(question)
print(answer)

For a headache, the type of medication you should take depends on the cause and severity of your headache. Here are some common types of over-the-counter (OTC) medications that can help with headaches:

1. **Acetaminophen (Tylenol)**: This is often effective for mild to moderate headaches. It works by reducing inflammation and pain.

2. **Ibuprofen (Advil, Motrin IB)**: This NSAID (non-steroidal anti-inflammatory drug) can be used for both mild to moderate headaches and also helps reduce inflammation. It's good for people who have conditions like arthritis or other inflammatory conditions.

3. **Aspirin**: Like acetaminophen and ibuprofen, aspirin can relieve headaches. It's particularly useful if you're prone to migraines and want to avoid NSAIDs.

4. **Paracetamol (Panadol)**: Similar to acetaminophen, this is another option for mild to moderate headaches.

If your headache is severe, persistent, or accompanied by other symptoms such as fever, vomiting, or neurological issues, it's i

##  RAG


In [5]:
from sentence_transformers import SentenceTransformer
import faiss

In [6]:
def load_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

In [7]:
def chunk_text(text, chunk_size=120, overlap=20):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [8]:
def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    embed_model = SentenceTransformer(model_name)
    embeddings = embed_model.encode(chunks, convert_to_numpy=True)
    return embed_model, embeddings

In [9]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [10]:
def search_index(query, embed_model, index, chunks, k=3):
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

### Build the index from our document
Upload `symptoms_and_medicines.txt` to the same folder as this notebook
(or update the path below) before running this cell.

In [13]:
doc_path = "./symptoms_and_medicines.txt"

text = load_text(doc_path)
chunks = chunk_text(text, chunk_size=120, overlap=20)

embed_model, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)

print(f"Document split into {len(chunks)} chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Document split into 198 chunks


### Ask the same question again — this time with retrieval

In [14]:
question = input('What are your symptom')

What are your symptomHeadache


In [15]:

top_chunks = search_index(question, embed_model, index, chunks, k=3)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
headache. Adult dose: 500-1000 mg every 4-6h, max 4000 mg/day. Side effects: Rare at normal doses; nausea. Precautions: Liver damage risk in overdose or with alcohol use; check other products for hidden acetaminophen. 3. Diclofenac Gel (Topical) - Category: Topical NSAID - Type: OTC. Uses: Muscle pain, joint pain, localized inflammation. Adult dose: Apply to affected area 3-4 times/day. Side effects: Local skin irritation. Precautions: Do not apply to broken skin; wash hands after use. 4. Methyl Salicylate Cream (Muscle Rub) - Category: Topical Analgesic (counterirritant) - Type: OTC. Uses: Muscle aches, joint pain, back pain. Adult dose: Apply to affected area up to 3-4 times/day. Side effects: Skin irritation, warming/burning sensation. Precautions: Do not use with heating pads; avoid

--- Chunk 2 ---
headache. Adult dose: 500-1000 mg every 4-6h, max 4000 mg/day. Side effects: Rare at normal doses; nausea. Precautions: Liver damage risk in overdose or with alcohol us

In [16]:
context = "\n\n".join(top_chunks)

prompt = f"""You are a pharmacist assistant. Using ONLY the reference information below,
answer the question. If the reference information does not contain the answer,
say clearly that you don't have this information in your knowledge base.

Reference information:
{context}

Question: {question}
"""

answer = generate_text(prompt, max_new_tokens=400)
print(answer)

For headaches, the appropriate dosage and precautions can vary depending on the specific medication used. However, based on the information provided, here is the relevant data:

**Dosage:** The dosage for treating headaches can be found under different medications listed. For **Paracetamol + Caffeine combo**, the adult dose is as per the product label, with a maximum of 4000 mg of paracetamol per day.

**Precautions:** 
- **Liver damage risk**: There is a risk of liver damage in overdose or when combined with alcohol.
- **Watch total paracetamol intake**: This is important because taking too much paracetamol over time can lead to liver damage.
- **Avoid extra caffeine sources**: Since caffeine is included in this combination, it's important to be cautious about consuming additional caffeine from other sources.
- **Jitteriness, insomnia**: These side effects may occur if taken late in the day due to the caffeine component.

Please note that these precautions are specific to the Paraceta

### Mentor mode test — asking about a medicine that is NOT in our document
This checks that the bot honestly says "I don't know" instead of making
something up.

In [17]:
question_2 = input('Tell me about the Medicine you wanna know about')

Tell me about the Medicine you wanna know aboutBananas


In [18]:

top_chunks_2 = search_index(question_2, embed_model, index, chunks, k=3)
context_2 = "\n\n".join(top_chunks_2)

prompt_2 = f"""You are a pharmacist assistant. Using ONLY the reference information below,
answer the question. If the reference information does not contain the answer,
say clearly that you don't have this information in your knowledge base.

Reference information:
{context_2}

Question: {question_2}
"""

answer_2 = generate_text(prompt_2, max_new_tokens=200)
print(answer_2)

I don't have this information in my knowledge base. The provided reference information does not mention bananas or any products related to bananas.


In [22]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import threading
import nest_asyncio
import uvicorn

nest_asyncio.apply()

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    prompt: str

In [23]:
def rag_answer(question: str):
    top_chunks = search_index(question, embed_model, index, chunks, k=3)

    context = "\n\n".join(top_chunks)

    prompt = f"""You are a pharmacist assistant. Using ONLY the reference information below,
answer the question. If the reference information does not contain the answer,
say clearly that you don't have this information in your knowledge base.

Reference information:
{context}

Question: {question}
"""

    return generate_text(prompt, max_new_tokens=400)

In [24]:
@app.post("/chat")
def chat(req: ChatRequest):
    return {
        "response": rag_answer(req.prompt)
    }


@app.get("/")
def root():
    return {
        "status": "running",
        "model": "Qwen2.5-3B-Instruct"
    }

In [25]:
def start_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=start_api, daemon=True).start()

print("✅ FastAPI running on http://localhost:8000")

✅ FastAPI running on http://localhost:8000


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8000

2026-07-09T15:33:25Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-09T15:33:25Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-09T15:33:30Z INF +--------------------------------------------------------------------------------------------+
2026-07-09T15:33:30Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-09T15:33:30Z INF |  https://issue-peak-shelter-cite.trycloudflare.com    